In [1]:
from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

## CFM 101 — Group Assignment 2026
### Robo-Advising Challenge: Two Clients, Two Strategies

**Team Number:** 11 
**Team Member Names (three or four):** three

Enter the names of all participating members below (one name per line):
- Brian
- Vansh
- Jenny 

**Submission:** Friday, November 27, 2026, at 9:00 AM (Eastern), via Dropbox.  
**Submit this completed Jupyter notebook only.**

Your team must construct **both** portfolios from the supplied `Tickers.csv` file:

- **Client A — Growth:** $\$1,000,000$ CAD; seek the highest expected return.
- **Client B — Low Risk:** $\$750,000$ CAD; seek the lowest portfolio volatility.

Both clients require 10–20 eligible stocks, at most 15% in any stock and at most 40% in any sector. Read the full assignment handout for all eligibility, fee, currency, and competition rules. Do not hardcode the selected stocks. Check that tickers are valid. The supplied ticker universe will permit feasible portfolios.

**No generative AI may be used in preparing this assignment.**

## Data preparation and shared functions

Read `Tickers.csv`, check ticker eligibility, retrieve the data needed for your strategies, and create any functions you will reuse for both clients. Use closing prices from **November 27, 2026** for the purchase calculations. All portfolio values and weights must be calculated in CAD, with purchase fees paid from the client budget.

Some global constants

In [2]:
# our group number
group = 11

# dates for filtering / data cleaning
filter_start = "2025-10-01"
filter_end = "2026-10-01"

# dates for historical analysis
period_start = "2023-10-01"
period_end = "2026-10-01"

# dates for competition
buy_start = "2026-11-27" # CHANGE BEFORE SUBMITTING,
day_after = "2026-11-28" # CHANGE BEFORE SUBMITTING,

# set budgets
budget_A = 1000000
budget_B = 750000

# Max weight in any one sector.
sector_limit = 0.40

# Max weight of any single stock
stock_limit = 0.15

# Min and max number of stocks
min_stocks = 10
max_stocks = 20

Functions to create final portfolio dataframes, and save them as csv file, giving back the budget reconciliation and fees too:

In [3]:
# fees is a function that takes in a number of shares
# it returns the fee accumulated for purchasing that number, in USD

def fees(shares):
    return min(2.15, 0.001 * shares)

# make_base_table is a function that takes in a portfolio dictionary (tickers are keys, values are weights) and a budget (client's A or B)
# it returns the base table (containing purchase prices and estimated fees), the total estimated fees, and the USD to CAD rate on purchase date
# this estimated fees is used to compute the remaining amount investable, and then find real fees (which will be slightly less)

def make_base_table(portfolio, budget):
    # get the tickers from the dictionary of target weights
    tickers = list(portfolio.keys())

    # download the adjusted closing prices for the purchase date
    data = yf.download(tickers, start = buy_start, end = day_after, threads=False, auto_adjust=True)["Close"]

    # get CAD value of one USD on purchase date
    usd_to_cad = yf.download("CAD=X", start = buy_start, end = day_after, threads=False, auto_adjust=True)["Close"].iloc[0,0]

    # create a dictionary to store table's values
    table = {
        "Ticker": [], 
        "Price": [], 
        "Currency": [], 
        "Weight": [], 
        "Price CAD": [],
        "Estimated Fees": []
    }

    # loop through each ticker
    for tick in tickers:
        # look up the purchase price and origina trading currency
        price = data.loc[buy_start, tick]
        currency = stock_info.loc[tick, "Currency"]

        # convert USD prices to CAD if needed
        if currency == "USD":
            price_cad = price * usd_to_cad
        else:
            price_cad = price

        # estimate shares using the full budget before deducting fee
        estimated_shares = portfolio[tick] * budget / price_cad
        # convert the estimated fee from USD to CAD, using estimated shares count
        estimated_fees = fees(estimated_shares) * usd_to_cad
        
        # store the current stock and its calculated values / information in the table dict
        table["Ticker"].append(tick)
        table["Price"].append(price)
        table["Currency"].append(currency)
        table["Weight"].append(portfolio[tick])
        table["Price CAD"].append(price_cad)
        table["Estimated Fees"].append(estimated_fees)

    # convert the dict, table, into dataframe
    table = pd.DataFrame(table)

    # return table, total estimated fee, and purchase date exchange rate
    return table, table["Estimated Fees"].sum(), usd_to_cad

In [4]:
# final_table is a function that takes in a portfolio dictionary (tickers are keys, values are weights) and a budget (client's A or B)
# it outputs the csv file for that portfolio, and returns the final table, the actual fees, and the total actual cost
# it calls base_table to get base estimate fees and costs for the portfolio, to calculate actual costs

def final_table(portfolio, budget):

    # create a base table missing some data, but calculating estimated fees
    # store values we need from that base table
    base_table, estimated_fees, usd_to_cad = make_base_table(portfolio, budget)

    tickers = list(portfolio.keys())
        
    table = base_table.copy()

    # reserve estimated fees
    investable = budget - estimated_fees
    # calculate fractional shares
    table["Shares"] = table["Weight"] * investable / table["Price CAD"]
    # calculate each holding value in CAD
    table["Value"] = table["Shares"] * table["Price CAD"]
    # recalculate weights using the invested stock value only
    table["Weight"] = table["Value"] / table["Value"].sum()

    actual_fees = 0

    for i in range(len(tickers)):
        actual_fees += fees(table.loc[i, "Shares"]) * usd_to_cad

    # Drop unneeded columns, and reorder columns
    table = table.drop(columns=["Price CAD", "Estimated Fees"])
    table = table[["Ticker", "Price", "Currency", "Shares", "Value", "Weight"]]

    # Calculate total CAD spent for budget reconciliation
    total_cost = table["Value"].sum() + actual_fees
    
    # Reindex final table
    table.index = range(1, len(table.index) + 1)
    
    # Create dataframe for csv file output
    file_table = table[["Ticker", "Shares"]]

    # Create option based on the budget to store the right kind of filename
    if budget == budget_A:
        option = "A"
    elif budget == budget_B:
        option = "B"

    # export ticker and share count without the df index
    file_table.to_csv(f"Stocks_Group_{group}_{option}.csv", index=False)
    
    return table, actual_fees, total_cost

Function to check that a portfolio is valid (to be run at the very end of portfolio creation):
Your checks must verify: valid/eligible tickers, stock count, maximum individual weight, maximum sector weight, weights summing to 100%, and total purchases plus fees within budget.

In [5]:
# validate_portfolio is a function that takes in a final portfolio dataframe (from final_table), its total fees, total costs, and the budget
# it outputs PASS or FAIL for the protfolio, for each of the checks described above

def validate_portfolio(table, actual_fees, total_cost, budget):

    # extract tickers from the final portfolio
    tickers = table["Ticker"]

    # 1.1 every ticker is valid
    # assume eligibility until a ticker fails
    eligible = True
    
    for tick in tickers:
        t = yf.Ticker(tick)

        # Check if there is data in here (i.e. not a delisted or invalid ticker)
        if t.history(period="5d").empty:
            print(tick, "doesn't exist")
            eligible = False
            break
    
        # 1.2 every ticker is eligible

        # if the ticker is not in stock info's index (filtered list of eligible stocks)
        if tick not in eligible_tickers:
            print(tick, "not eligible")
            eligible = False
            break

    if eligible:
        print("PASS: tickers valid and eligible")
    else:
        print("FAIL: some tickers invalid / ineligible")


    # 2 valid stock count

    # count rows
    count = len(table.index)

    if count >= min_stocks and count <= max_stocks:
        print("PASS: valid stock count")
    else:
        print("FAIL: invalid stock count")

    # 3 and 4, maximum individual weight and weights sum

    weights_ok = True
    weights_sum = 0

    for i in table.index:
        # recalculate each weight from the actual CAD holding value
        weight = table.loc[i, "Value"] / table["Value"].sum()
        weights_sum += weight

        # allow for tiny rounding errors
        if weight > stock_limit and not np.isclose(weight, stock_limit):
            weights_ok = False
            

    if weights_ok:
        print("PASS: max stock weight")
    else:
        print("FAIL: a stock is over the max weight")

    if np.isclose(weights_sum, 1):
        print("PASS: weights sum to 100%")
    else:
        print("FAIL: weights don't sum to 100%")

    # 5 maximum sector weight

    sectors_ok = True
    sector_weights = {}

    # initialize sector weights
    for tick in table["Ticker"].tolist():
        sector_weights[stock_info.loc[tick, "Sector"]] = 0

    for i in table.index:
        sector = stock_info.loc[table.loc[i, "Ticker"], "Sector"]
        sector_weights[sector] += table.loc[i, "Value"] / table["Value"].sum()

    # check each sector total against the sector limit
    for sector in sector_weights:
        if sector_weights[sector] > sector_limit and not np.isclose(sector_weights[sector], sector_limit):
            sectors_ok = False
            

    if sectors_ok:
        print("PASS: valid sector weights")
    else:
        print("FAIL: invalid sector weights")

    # 6 total purchase cost and fees within budget

    # allow for tiny rounding error, 1 singular cent
    if total_cost <= budget + 0.01:
        print("PASS: purchases within budget")
    else:
        print("FAIL: purchases not within budget")
        
    # show the actual fee and total spending in CAD, and budget reconciliation
    print(f"Stock Purchases: ${table['Value'].sum():.2f} CAD")
    print(f"Actual Fees: ${actual_fees:.2f} CAD")
    print(f"Total Cost: ${total_cost:.2f} CAD")
    

Extracting stocks / Data Cleaning:

In [6]:
# renamed ticker_file
ticker_file = "Tickers.csv"
raw_tickers = pd.read_csv(ticker_file, header=None, names=["Ticker"]) 
# deletes the repeated tickers
tickers = raw_tickers["Ticker"].drop_duplicates().tolist()  

stock_rows = []

for tick in tickers:
    t = yf.Ticker(tick)
    # get data for market, currency, sector and market cap
    information = t.get_info()

    # Check if there is data in here (i.e. not a delisted or invalid ticker)
    if t.history(period='max').empty:
        print(tick, "doesn't exist")
        continue
    
    # Check if it's a CAD or USD stock traded in CA or US market
    if information.get("market") not in ["ca_market", "us_market"]:
        print(tick, "wrong market")
        continue

    # Check if it's a stock and not something like a bond or index fund
    if information.get("quoteType") != "EQUITY":
        print(tick, "not stock")
        continue

    # check currency
    if information.get("currency") not in ["CAD", "USD"]:
        print(tick, "currency wrong")
        continue

    # check sector info
    if not information.get("sector"):
        print(tick, "sector missing")
        continue

    # save stock info for later processing
    stock_rows.append({
            "Ticker": tick,
            "Currency": information.get("currency"),
            "Sector": information.get("sector"),
            "Market Cap": information.get("marketCap")
        })

stock_info = pd.DataFrame(stock_rows).set_index("Ticker") 
# set ticker as index
listed_tickers = stock_info.index.tolist()
# save the list for filter
display(stock_info.head())

$AGN: possibly delisted; no timezone found


AGN doesn't exist


$BK: possibly delisted; no price data found  (1d 1927-12-22 -> 2026-11-27)


BK doesn't exist


$CELG: possibly delisted; no timezone found


CELG doesn't exist


$MON: possibly delisted; no timezone found


MON doesn't exist


$RTN: possibly delisted; no timezone found


RTN doesn't exist


,Currency,Sector,Market Cap
Ticker,,,
AAPL,USD,Technology,5167353167872
ABBV,USD,Healthcare,520527642624
ABT,USD,Healthcare,187049689088
ACN,USD,Technology,142066450432
AIG,USD,Financial Services,42631229440


In [7]:
# download stock prices, trading volumes and exchange rates
history = yf.download(
    listed_tickers + ["CAD=X"],
    start=period_start,
    end=period_end,
    threads=False,
    auto_adjust=True
)

# download close, volume and foreign exchange info
close_prices = history["Close"][listed_tickers]
# get closing price info
volume_data = history["Volume"][listed_tickers]
# get volume info
cad_per_usd = history["Close"]["CAD=X"].dropna()
# get USD to CAD exchange info

[*********************100%***********************]  36 of 36 completed


In [8]:
# create a seperate volume data for filter
volume_data = volume_data.loc[(volume_data.index >= filter_start) & (volume_data.index < filter_end)]
print(f"volume screening period: {volume_data.index.min()}, {volume_data.index.max()}")

# Note:
# average daily trading volume = total volume traded over a period (volume) / # of trading dasy in the period
# since months with fewer than 18 valid trading days may affect the average daily trading volume
# 1. filter out months with fewer than 18 valid trading days before calculating liquidity
# 2. use only eligible months to calculate liquidity and determine eligible_tickers

eligible_tickers = []
average_volume = {} 

for ticker in listed_tickers:
    volume = volume_data[ticker].dropna()  
    # delete missed value
    # collect daily volumes only from valid months
    volume_days = []

    # group each stock's volume by calendar month
    for month, monthly_volume in volume.groupby(volume.index.to_period("M")): 
        if len(monthly_volume) >= 18:  
            volume_days.extend(monthly_volume)  

    average_volume[ticker] = pd.Series(volume_days).mean()  
    # transfer list to series to calculate average trading volume
    # keep stocks meeting the minimum average daily volume requirment
    if average_volume[ticker] >= 5_000:
        eligible_tickers.append(ticker)

# attach the calculated liquidity stats by ticker
stock_info["Average Daily Volume"] = pd.Series(average_volume)  
print(f"{len(eligible_tickers)} eligible stocks.")

display(stock_info["Average Daily Volume"].head())

volume screening period: 2025-10-01 00:00:00, 2026-09-30 00:00:00
35 eligible stocks.


Ticker
AAPL    4.828203e+07
ABBV    6.281745e+06
ABT     1.008423e+07
ACN     6.116120e+06
AIG     4.324195e+06
Name: Average Daily Volume, dtype: float64

In [9]:
# tomake sure all stocks are using the same currency

# convert prices to CAD
cad_prices = close_prices[eligible_tickers].copy()
for ticker in eligible_tickers:
    if stock_info.loc[ticker, "Currency"] == "USD":
        # if currency is USD, then change to CAD
        cad_prices[ticker] = cad_prices[ticker] * cad_per_usd
        # match closing price and exchange rate by date

cad_prices = cad_prices.dropna()  
# delete dates which missing info

In [10]:
# After all filtering, this is the table that contains all the complete data for the full consideration period, and valid stocks
# No CBIL.TO in this

display(cad_prices.head())

Ticker,AAPL,ABBV,ABT,ACN,AIG,AMZN,AXP,BA,BAC,BB.TO,...,PYPL,QCOM,SHOP.TO,T.TO,TD.TO,TXN,UNH,UNP,UPS,USB
Date,,,,,,,,,,,,,,,,,,,,,
2023-10-02,232.519125,180.319200,121.325716,393.013051,76.241147,175.725132,195.350744,254.954813,33.728010,6.02,...,78.610926,141.522601,73.839996,17.100691,70.040115,198.282192,655.916213,257.465500,174.719966,38.394420
2023-10-03,232.456428,180.677267,122.319401,393.376058,75.013737,170.570816,190.994155,258.372658,32.977465,5.98,...,77.500919,141.424380,70.440002,17.204714,69.378120,196.408601,654.324061,260.870999,174.530872,37.721729
2023-10-04,234.716907,181.430017,122.433567,398.388518,76.116891,174.105576,193.537665,255.990027,33.094827,5.86,...,79.408646,141.326129,72.230003,17.756866,69.195213,197.781115,657.373876,259.559004,174.915339,37.691547
2023-10-05,237.011665,181.598919,123.452851,399.757000,77.327376,173.121948,195.441625,256.040701,33.333050,5.20,...,78.130362,141.623515,71.510002,18.100964,69.997551,194.697581,666.297682,258.126533,174.975284,38.550101
2023-10-06,239.833951,182.060442,123.977228,401.894206,78.033789,175.378129,197.848890,256.817403,33.252435,5.15,...,78.304584,142.615900,73.269997,18.044949,70.359047,196.517888,675.474447,259.738355,175.497617,38.430053


## Non Data Filtering Variables and Functions to be Used in Both Portfolios:

In [11]:
# use S&P 500 as the market benchmark
market_history = yf.download("^GSPC", start=period_start, end=period_end, threads=False, auto_adjust=True)["Close"]

# convert the market index series
market_cad = (market_history["^GSPC"] * cad_per_usd).to_frame("Market")

# drop days where mismatch in market exchange and S&P500 availability
market_cad = market_cad.dropna()

# Merge SP500 and valid tickers dataframes, dropping any dates with mismatch in markets
aligned_prices = pd.concat([cad_prices[eligible_tickers], market_cad], axis=1, join="inner").dropna()
# Calculate returns, dropping days (list first day) with no valid calculation
aligned_returns = aligned_prices.pct_change().iloc[1:].dropna()

# Aligned_prices is used for closing price data for all subsequent calculations
display(aligned_prices.head())
# Aligned_returns is used for daily return data for all subsequent calculations
display(aligned_returns.head())

[*********************100%***********************]  1 of 1 completed


,AAPL,ABBV,ABT,ACN,AIG,AMZN,AXP,BA,BAC,BB.TO,...,QCOM,SHOP.TO,T.TO,TD.TO,TXN,UNH,UNP,UPS,USB,Market
Date,,,,,,,,,,,,,,,,,,,,,
2023-10-02,232.519125,180.319200,121.325716,393.013051,76.241147,175.725132,195.350744,254.954813,33.728010,6.02,...,141.522601,73.839996,17.100691,70.040115,198.282192,655.916213,257.465500,174.719966,38.394420,5820.932201
2023-10-03,232.456428,180.677267,122.319401,393.376058,75.013737,170.570816,190.994155,258.372658,32.977465,5.98,...,141.424380,70.440002,17.204714,69.378120,196.408601,654.324061,260.870999,174.530872,37.721729,5784.322991
2023-10-04,234.716907,181.430017,122.433567,398.388518,76.116891,174.105576,193.537665,255.990027,33.094827,5.86,...,141.326129,72.230003,17.756866,69.195213,197.781115,657.373876,259.559004,174.915339,37.691547,5845.217719
2023-10-05,237.011665,181.598919,123.452851,399.757000,77.327376,173.121948,195.441625,256.040701,33.333050,5.20,...,141.623515,71.510002,18.100964,69.997551,194.697581,666.297682,258.126533,174.975284,38.550101,5852.541619
2023-10-06,239.833951,182.060442,123.977228,401.894206,78.033789,175.378129,197.848890,256.817403,33.252435,5.15,...,142.615900,73.269997,18.044949,70.359047,196.517888,675.474447,259.738355,175.497617,38.430053,5905.100605


,AAPL,ABBV,ABT,ACN,AIG,AMZN,AXP,BA,BAC,BB.TO,...,QCOM,SHOP.TO,T.TO,TD.TO,TXN,UNH,UNP,UPS,USB,Market
Date,,,,,,,,,,,,,,,,,,,,,
2023-10-03,-0.000270,0.001986,0.008190,0.000924,-0.016099,-0.029332,-0.022301,0.013406,-0.022253,-0.006645,...,-0.000694,-0.046045,0.006083,-0.009452,-0.009449,-0.002427,0.013227,-0.001082,-0.017521,-0.006289
2023-10-04,0.009724,0.004166,0.000933,0.012742,0.014706,0.020723,0.013317,-0.009222,0.003559,-0.020067,...,-0.000695,0.025412,0.032093,-0.002636,0.006988,0.004661,-0.005029,0.002203,-0.000800,0.010528
2023-10-05,0.009777,0.000931,0.008325,0.003435,0.015903,-0.005650,0.009838,0.000198,0.007198,-0.112628,...,0.002104,-0.009968,0.019378,0.011595,-0.015591,0.013575,-0.005519,0.000343,0.022778,0.001253
2023-10-06,0.011908,0.002541,0.004248,0.005346,0.009135,0.013032,0.012317,0.003034,-0.002418,-0.009615,...,0.007007,0.024612,-0.003095,0.005164,0.009349,0.013773,0.006244,0.002985,-0.003114,0.008981
2023-10-10,-0.004110,-0.004790,-0.002180,-0.008722,-0.004251,0.002635,0.001549,0.023386,0.026592,-0.048544,...,0.003644,0.020882,0.005321,0.009022,-0.004970,-0.010211,0.009903,-0.000914,0.013568,0.002304


In [12]:
# z_score is a function that takes a Series of a statistic and whether a higher value is better for ranking stocks
# it returns a Series of each stock's z score, with missing values given the worst z score (depends on whether higher or lower is worse)

def z_score(ser, higher_better):
    # standardize the statistic relative to its mean and std
    z = (ser-ser.mean())/ser.std()
    
    # replace missing values with the worst available z-score
    if higher_better:
        return z.fillna(z.min())
    else:
        return z.fillna(z.max())

In [13]:
# compare_equal_weight is a function that takes a Series (indexes are tickers, values hold weights)
# it constructs the same portfolio but equal weighted, and gives volatiltiy and return % comparisons over our historical sample period

def compare_equal_weight(weights):
    # Equal-weight Comparison
    
    # calculate the price growth factor for each stock
    prices = cad_prices[weights.index] # store our portfolio's stocks into prices DataFrame
    growth = prices.iloc[-1] / prices.iloc[0]
    
    equal_weights = pd.Series(1 / len(weights), index = weights.index)
    
    # calculate cumulative historical returns
    strategy_return = (growth * weights).sum() - 1
    equal_return = (growth * equal_weights).sum() - 1
    
    # calculate daily volatility
    strategy_std = weighted_std(weights, prices)
    equal_std = weighted_std(equal_weights, prices)
    
    # comparison table
    comparison = pd.DataFrame({
        "Historical Return (%)": [100 * strategy_return, 100 * equal_return],
        "Daily Std (%)": [100 * strategy_std, 100 * equal_std],
    }, index=["Optimized Portfolio", "Equal Weight Portfolio"])
    
    display(comparison.round(6))
    
    # produce the differences in percentage points
    print(f"Start Date: {prices.index[0].date()}")
    print(f"End Date: {prices.index[-1].date()}")
    print(f"Dates are inclusive")
    print(f"Daily Std change: {100 * (strategy_std - equal_std):+.4f} percentage points")
    print(f"Historical return change: {100 * (strategy_return - equal_return):+.4f} percentage points")

In [14]:
# count_price_plummets is a function that takes in a period length (in trading days) and a price drop threshold (decimal)
# it divides our historical sample period into small chunks with the given period length
# then it returns a Dataframe containing in how many time each stock dropped by the threshold, in any of those small period chunks (rough tally of short term crash in price)

def count_price_plummets(period, threshold):
    # create a table to store the number of detected declines of each stock
    result = pd.DataFrame()
    for tick in eligible_tickers:
        result.loc[tick, "Occurrences"] = 0
    # start with most recent complete price interval
    interval = [len(cad_prices)-period-1, len(cad_prices)-1]
    while interval[0] >= 0:
        for tick in eligible_tickers:
            # compare the ending price with the threshold times the starting price
            if cad_prices.iloc[interval[1]][tick] <= cad_prices.iloc[interval[0]][tick] * threshold:
                result.loc[tick, "Occurrences"]+=1

        # shift endpoints
        interval[0]-=period
        interval[1]-=period

    return result

In [15]:
# weighted_std is a function that takes in starting portfolio (Series, indexing are tickers, values are weights) and a table of closing prices for each stock
# it returns the std of the daily returns of that buy-and-hold portfolio over the dates in the price table

def weighted_std(weights, period_prices): 
        # shares bought with each stock's weight on the first day
        shares = weights / period_prices.iloc[0] 
        # total portfolio value each day
        values = (period_prices * shares).sum(axis=1)
        # daily % change in portfolio value (first day has no return so drop it)
        returns = values.pct_change().iloc[1:]
        # return std of the daily returns
        return returns.std(ddof=1)

## CLIENT A - HIGH RETURNS PORTFOLIO:

In [16]:
# make a separate copy of CAD prices for high-return calculations
full_data = cad_prices.copy()

# add the treasury bill ETF for estimating the risk-free return
full_data["CBIL.TO"] = yf.download("CBIL.TO", period_start, period_end, auto_adjust=True)["Close"]

# if days are missing, replace them with the previous day's price
full_data = full_data.ffill()

# show the first few rows to check the data
display(full_data.head())

[*********************100%***********************]  1 of 1 completed


Ticker,AAPL,ABBV,ABT,ACN,AIG,AMZN,AXP,BA,BAC,BB.TO,...,QCOM,SHOP.TO,T.TO,TD.TO,TXN,UNH,UNP,UPS,USB,CBIL.TO
Date,,,,,,,,,,,,,,,,,,,,,
2023-10-02,232.519125,180.319200,121.325716,393.013051,76.241147,175.725132,195.350744,254.954813,33.728010,6.02,...,141.522601,73.839996,17.100691,70.040115,198.282192,655.916213,257.465500,174.719966,38.394420,45.243134
2023-10-03,232.456428,180.677267,122.319401,393.376058,75.013737,170.570816,190.994155,258.372658,32.977465,5.98,...,141.424380,70.440002,17.204714,69.378120,196.408601,654.324061,260.870999,174.530872,37.721729,45.229580
2023-10-04,234.716907,181.430017,122.433567,398.388518,76.116891,174.105576,193.537665,255.990027,33.094827,5.86,...,141.326129,72.230003,17.756866,69.195213,197.781115,657.373876,259.559004,174.915339,37.691547,45.243134
2023-10-05,237.011665,181.598919,123.452851,399.757000,77.327376,173.121948,195.441625,256.040701,33.333050,5.20,...,141.623515,71.510002,18.100964,69.997551,194.697581,666.297682,258.126533,174.975284,38.550101,45.247658
2023-10-06,239.833951,182.060442,123.977228,401.894206,78.033789,175.378129,197.848890,256.817403,33.252435,5.15,...,142.615900,73.269997,18.044949,70.359047,196.517888,675.474447,259.738355,175.497617,38.430053,45.256702


In [17]:
# half life for sharpe calculations is 1 trading year = roughly 252 days
half_life = 252

# since dividend calculation function works in calendar days, adjust half life to be 1 year = 365 calendar days
dividend_half_life = 365

In [18]:
# get_stats is a function that takes in a ticker and a half life
# it returns the half-life weighted mean and std of the stock's daily returns, i.e. older makes less impact on statistic returned

def get_stats(tick, half_life):
    # get available price history for the stock
    s = full_data[tick].dropna()
    # calculate daily logarithmics returns, weighting recent days more using our half life constant
    delta = np.log(1 + s.pct_change()).ewm(halflife = half_life)
    # take the latest weighted std and mean
    standard_deviation = delta.std().iloc[-1]
    average = delta.mean().iloc[-1]

    return average, standard_deviation

Calculating statistics for stocks:

In [19]:
# get the risk-free return rate (from the ETF of T-Bills, CBIL.TO)
Rf = get_stats("CBIL.TO", half_life)[0]
print("Risk free return:", Rf)

# create a dict to store sharpe of each stock over sample period
sharpe = {}

# calculate Sharpe ratio for each eligible stock
for tick in eligible_tickers:
    # get the stock's weighted mean return and std
    stats = get_stats(tick, half_life)
    # Sharpe ratio = (mean return - risk-free return) / std
    # this creates a Sharpe ratio that is effectively recency weighted 
    sharpe[tick] = (stats[0] - Rf)/stats[1]

# sort the stocks by Sharpe ratios
sharpe = dict(sorted(sharpe.items(), key = lambda item:item[1]))
print(sharpe)

Risk free return: 0.00011224395239244361
{'T.TO': np.float64(-0.09195378293620031), 'ACN': np.float64(-0.035911999243616544), 'PEP': np.float64(-0.032150657871847475), 'UPS': np.float64(-0.02036301915612949), 'ABT': np.float64(-0.017422250196516265), 'PYPL': np.float64(-0.014259053284670179), 'BA': np.float64(-0.012571456924862418), 'UNH': np.float64(-0.009529172377901367), 'CL': np.float64(0.003895268782411308), 'AIG': np.float64(0.004436267908136788), 'LMT': np.float64(0.008668861844633918), 'AXP': np.float64(0.017961986738669614), 'QCOM': np.float64(0.018528735674174828), 'BLK': np.float64(0.024127753884079346), 'PFE': np.float64(0.02595920382673528), 'BIIB': np.float64(0.027516699796933473), 'AMZN': np.float64(0.02788413331347568), 'SHOP.TO': np.float64(0.029921343308697897), 'UNP': np.float64(0.031031977178862396), 'BMY': np.float64(0.0317425915746662), 'LLY': np.float64(0.04066871695723559), 'TXN': np.float64(0.04228451596872343), 'BAC': np.float64(0.04250234265172469), 'USB': np

In [20]:
# get_momentum is a function that takes in no parameters
# it returns a Series (indexing of tickers, values of returns) of each stock's returns over 12 months (imagine a 12 month slice of time ending when our historical sample ends, but then shift it back by 1 month) 
# this is a 12-1 momentum calculation 

def get_momentum():
    # use CAD prices of eligible stocks only
    prices = full_data[eligible_tickers]

    # convert the period_end text into datetime object
    end = pd.Timestamp(period_end)
    # subtract 1 month to get end for momentum calculation (skipping last month because of reversal)
    momentum_end = end.replace(month = end.month - 1)
    # take the above and subtract 1 year to get the start for the momentum calculation
    momentum_start = momentum_end.replace(year = momentum_end.year - 1)
    
    # calculate the price return between the two dates
    return (prices.asof(momentum_end)) / (prices.asof(momentum_start)) - 1

In [21]:
# get_dividends is a function that takes in no parameters
# it returns a Series of each stock's half-life weighted dividend yield (dividends paid / price) over our historical sample
def get_dividends():
    # store the weighted dividend yields by ticker
    result = pd.Series()
    for tick in eligible_tickers:
        div = yf.Ticker(tick).dividends
        # match date with price data
        div.index = div.index.tz_localize(None).normalize()

        # keep dividend payments within historical period
        div = div[(div.index >= period_start) & (div.index < period_end)]

        # stocks with 0 dividends get a dividend yield rating of 0
        if div.empty:
            result[tick] = 0
            continue

        # convert USD dividends to CAD
        if stock_info.loc[tick, "Currency"] == "USD":
            div = (div * cad_per_usd.asof(div.index))

        # get the latest available stock price at each dividend date
        price = full_data[tick].asof(div.index)

        # our stat "dividend yield" is calculated as dividend / price
        yields = div/price

        # give older dividend payments less weight
        age = (pd.Timestamp(filter_end) - div.index).days
        weights = 0.5**(age/dividend_half_life)

        # add up the weighted dividend yields
        result[tick] = np.sum(yields*weights)

    return result

In [22]:
# sharpe_check is a function that takes in no parameters
# it returns a Series of each stock's Sharpe ratio if it is negative, and 0 if it is positive
# so we can penalize only negative ones
def sharpe_check():
    result = pd.Series()
    for tick in eligible_tickers:
        # keep negative Sharpe ratios and replace positive ones with zero
        result[tick] = min(0, sharpe[tick])

    return result

In [23]:
# revenue_CAGR is a function that takes in no parameters
# it returns a Series of each stock's compound annual growth rate of total revenue (comparing the latest and oldest revenue reporting from our historical sample)
def revenue_CAGR():
    result = pd.Series()
    for tick in eligible_tickers:
        # get the annual income statement for this stock
        stmt = yf.Ticker(tick).income_stmt
        if "Total Revenue" not in stmt.index:
            result[tick] = np.nan
            continue

        # sort the available revenue data from oldest to newest
        rev = stmt.loc["Total Revenue"].dropna().sort_index()

        # need at least two revenue figures to calculate growth
        if len(rev) < 2:
            result[tick] = np.nan
            continue

        # calculate the number of years between the first and last revenue dates
        years = (rev.index[-1] - rev.index[0]).days/365.25
        # calculate annual compounded revenue growth rate
        result[tick] = pow((rev.iloc[-1]/rev.iloc[0]), (1/years)) - 1

    return result

In [24]:
# revenue_std is a function that takes in no paramteres
# it returns a Series of the std of each stock's yearly revenue growth rates (volatility in their revenue growth)
def revenue_std():
    result = pd.Series()
    for tick in eligible_tickers:
        # get annual revenue data
        stmt = yf.Ticker(tick).income_stmt
        if "Total Revenue" not in stmt.index:
            result[tick] = np.nan
            continue
        
        # calculate the percentage change in revenue each year
        rev_delta = stmt.loc["Total Revenue"].dropna().sort_index().pct_change().dropna()

        # determine how much revenue growth varies
        result[tick] = rev_delta.std()
        # pull 3 yearly growth rates for more menaingful std
        if len(rev_delta) <= 2:
            result[tick] = np.nan

    return result

In [25]:
score_df = pd.DataFrame()

# we put True if a higher value of the statistic is better, and False if a lower value is better (decides what the default score given missing data should be)
# eg. dividends should be -ve because higher dividends means less growth
# count price drops of at least 20% within 15 trading day periods of market trading across historical sample
score_df["Drops"] = count_price_plummets(15, 0.8)["Occurrences"]
score_df["Momentum"] = z_score(get_momentum(), True)
score_df["Dividends"] = z_score(get_dividends(), False)
score_df["Sharpe Check"] = z_score(sharpe_check(), True)
score_df["Revenue CAGR"] = z_score(revenue_CAGR(), True)
score_df["Revenue STD"] = z_score(revenue_std(), False)

# note, some things have to be negative because they lower the score (this weight/factor is a multiplier on the z score / occurence rate)
score_weights = {"Drops": -1.5, "Momentum": 1, "Dividends": -0.375, "Sharpe Check": 0.4, "Revenue CAGR": 0.875, "Revenue STD": -0.25}

# multiply each statistic by its assigned weight and add them up for each stock
overall_score = sum(score_df[x] * score_weights[x] for x in score_weights).to_dict()

In [26]:
# sort stocks from highest to lowest growth scores
overall_score = dict(sorted(overall_score.items(), key=lambda x: -x[1]))

overall_score

{'LLY': 4.2104390883514125,
 'SHOP.TO': 2.344258421676694,
 'CAT': 2.2204471134630035,
 'TD.TO': 1.9011529865675572,
 'MRK': 1.7698361187648193,
 'RY.TO': 1.3705515151330947,
 'AAPL': 1.302424912815288,
 'AMZN': 1.0929164607000121,
 'C': 0.757008026359964,
 'BAC': 0.42822107838403234,
 'AXP': 0.3805269799529826,
 'KO': 0.28541535830787307,
 'PM': 0.26911557884001625,
 'BMY': 0.23271414550924002,
 'BLK': 0.01697207021485643,
 'UNP': 0.011859335500487816,
 'USB': -0.012475012854686482,
 'BIIB': -0.23065535590049138,
 'CL': -0.24539042944209416,
 'ABBV': -0.2474815059631201,
 'QCOM': -0.2670717797325115,
 'BA': -0.8385185380829333,
 'TXN': -0.9601167209067771,
 'LMT': -1.3223118320641438,
 'AIG': -1.5310367366373827,
 'MO': -1.6201107527982712,
 'PEP': -1.77091662458958,
 'UPS': -1.8823877062183911,
 'ABT': -1.9221854578576003,
 'ACN': -2.0316318283700734,
 'UNH': -2.0625768133783997,
 'PYPL': -2.5583788492502673,
 'BB.TO': -2.788277786691639,
 'PFE': -3.3483395863368535,
 'T.TO': -4.9539

In [27]:
# store number of stocks for our portfolio, total
# we will only take 10 stocks, 11th will serve as baseline for readjusting growth scores once we make them all positive
num_stocks = 11

# max number of stocks in each sector
max_per_sector_A = 3

Weighting:

In [28]:
# get_best_stocks is a function that takes in no parameters
# it returns a dict (keys are tickers, values are the scores) of the highest scoring tickers and their growth scores
# this number is capped to num_stocks
# the number of stocks per sector is capped to max_per_sector_A

def get_best_stocks():
    # keep track of the stocks chosen from each sector
    sector_chosen = {}

    # used to store the selected tickers and their growth scores
    chosen_stocks = {}

    chosen = 0
    
    # go through stocks from highest to lowest growth scores
    for tick, score in overall_score.items():
        sector = stock_info.loc[tick, "Sector"]

        # stop once we have enough candidates
        if chosen >= num_stocks:
            break
        
        # skip sectors that already have the max number of chosen stock
        if (sector in sector_chosen) and len(sector_chosen[sector]) == max_per_sector_A:
            continue
        else:
            if sector not in sector_chosen:
                sector_chosen[sector] = [tick]
            else:
                sector_chosen[sector].append(tick)

            # save the score and update the number of selected stocks
            chosen_stocks[tick] = score
            chosen+=1

    return chosen_stocks

chosen_stocks = get_best_stocks()

In [29]:
# use the lowest selected score 11th stock as the baseline to shift/reduce the rest of the scores by
# this will make the 11th have a 0% weight so we drop it, getting back to 10 stocks in portfolio
min_val = min(chosen_stocks.values())
shifted = {}

# shift scores so the 11th stock is scored 0 and all others are positive
for tick in chosen_stocks:
    shifted[tick] = chosen_stocks[tick] - min_val

# get_weights is a function that takes in a dict of stocks we chose for portfolio and their growth scores
# it returns a dict of each ticker's weight for the portfolio, in proportion to its shifted growth score (11th stock starts at score 0)
# also ensures stock and sector limtis are met

def get_weights(chosen_stocks):
    # sum the shifted scores
    total_weight = sum(shifted[x] for x in chosen_stocks)

    # start with 100% of the portfolio weight still available
    total_budget = 1.0

    value = {}

    # track weight assigned to each sector
    sector_weights = {}
    
    for tick in chosen_stocks:
        # stop once all the shifted score has been allocated, then only the 11th stock is left with score of 0
        if total_weight <= 0:
            break
            
        # give each stock its portion of the remaining allocatable weight, ensuring stock and sector limits are mainteined
        value[tick] = min(stock_limit, total_budget * shifted[tick]/total_weight, sector_limit - sector_weights.get(stock_info.loc[tick, "Sector"], 0))

        # update sector weight
        sector_weights[stock_info.loc[tick, "Sector"]] = sector_weights.get(stock_info.loc[tick, "Sector"], 0) + value[tick]

        # remove the current score and allocated weight before moving to the next stock
        total_weight -= shifted[tick]
        total_budget -= value[tick]

    return value

# convert dictionary into a Series indexed by ticker
weights_A = pd.Series(get_weights(chosen_stocks))

# Dropping the 11th stock with 0% weight
weights_A = weights_A[weights_A > 0]

## Outputting final Portfolio A

In [30]:
# send all data to function to create final dataframe
Portfolio_Final_A, fees_A, total_cost_A = final_table(weights_A.to_dict(), budget_A)

display(Portfolio_Final_A)

# check the final high-return portfolio restrictions
validate_portfolio(Portfolio_Final_A, fees_A, total_cost_A, budget_A)

$MRK: possibly delisted; no price data found  (1d 2026-11-27 -> 2026-11-28)
[                       0%                       ]$AMZN: possibly delisted; no price data found  (1d 2026-11-27 -> 2026-11-28)
[**************        30%                       ]  3 of 10 completed$CAT: possibly delisted; no price data found  (1d 2026-11-27 -> 2026-11-28)
[*******************   40%                       ]  4 of 10 completed$AAPL: possibly delisted; no price data found  (1d 2026-11-27 -> 2026-11-28)
[**********************50%                       ]  5 of 10 completed$LLY: possibly delisted; no price data found  (1d 2026-11-27 -> 2026-11-28)
[**********************60%****                   ]  6 of 10 completed$C: possibly delisted; no price data found  (1d 2026-11-27 -> 2026-11-28)
[**********************70%*********              ]  7 of 10 completed$KO: possibly delisted; no price data found  (1d 2026-11-27 -> 2026-11-28)
[*********************100%***********************]  10 of 10 completed

7 

KeyError: '2026-11-27'

### Client A — Financial justification
The code right below is for generating the equal weight portfolio for the same stocks chosen, also other stats to compare to.

In [ ]:
# compare the high-return weights against equal weights for the same tickers
compare_equal_weight(weights_A)

In [ ]:
# S&P 500 return and daily std over the same period, in CAD
market = aligned_prices["Market"]

# price on the first and last day of our sample
start_price = market.iloc[0]
end_price = market.iloc[-1]

# calculate return over the period
market_return = (end_price - start_price) / start_price

# daily std of S&P 500's returns
market_std = market.pct_change().std()

print(f"S&P 500 return: {100 * market_return:.2f}%")
print(f"S&P 500 daily std: {100 * market_std:.4f}%")

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.


Client A wants the highest expected return. Since future returns can't be predicted accurately by optimizing on past data, our strategy scores every eligible stock on current/historical signs of continued future growth, using more fundamentals and forward looking metrics to pick the highest scoring stocks, and gives more weight to the higher growth potential stocks. 

This is how we select stocks:


#### Stage 1 - Collect Needed Data

The period chosen for our historical sampling/data was: 2023-10-02 to 2026-09-30
We chose this because the client was a long-term growth strategy. This spans 3 years, and so gives a longer-term picture for statistics we will be using. Also, going further back, we noticed that some stocks were missing data we needed, some may not exist, and their businesses may not be similar to what they are today. 3 years is a good middle ground for how much data we can consider, and it's the same justification for the low risk portfolio using the same sampling period.

Notably, we download a treasury bill ETF (CBIL.TO), which stands in for the risk-free rate that we compare every stock against.

#### Stage 2 - Metrics for every stock

We calcualte 6 metrics, each rewarding or penalizing a stock's overall growth score:

### Metric 1: Number of drops

This is a future-predictive statistic we created. While std already looks at volatility, we specifically want to capture company risks that decrease value quickly. If a stock loses 20% within a few weeks due to company-specific news or shocks, such as earnings reports, similar events could happen again in the future. This accounts for non-market / business risk.

We split our sample into 15 trading day periods (about 3 trading weeks) and count how many times the stock fell by 20% or more within one of those periods. Stocks that dropped suddenly more often in the past have faced more company-specific news or shocks, such as earnings reports or regulation.

If the market as a whole declines sharply within 3 weeks in our sample, many stocks are penalized at once, so no single stock is shown as having a better growth score. However, if a specific stock or industry declines sharply while others don't, our penalty helps differentiate which stocks to pick. This directly captures non-market risks like poor management decisions or shocks from news, which can characterize certain stocks and industries since they occur more frequently for them than for others.

Each drop decreases the score by 1.5 units, because when we invest in stocks we want consistent growth. A stock with a history of plummeting in price can't be expected to have a stable long-term growth pattern, which is why we weigh each drop so harshly.


### Metric 2: Price momentum / rate of return

Our source for this stat is Investopedia:
Investopedia. (2021, March 4). Momentum indicates stock price strength. Investopedia. https://www.investopedia.com/articles/technical/081501.asp

Momentum measures how much a stock's price has risen over the past year. It's our most heavily weighted metric since it's a well used trading strategy for growth, and it's a useful indicator of the strength of a stock's price trend (Investopedia, 2021), and we want stocks that would continue in a strong upward trend. By Investopedia it's calculated as:

> Latest Price - Historical Price

However, this formula measures the change in dollars, which can't be compared across stocks with very different prices (eg. Berkshire Hathaway would be an outlier). The same dollar change means something completely different depending on the stock's price magnitude. So, we adjusted the raw dollar differences into a percentage return, to turn it into a comparative statistic (i.e. just rate of return):

> Momentum = (Latest Price − Historical Price) / Historical Price = (Latest Price / Historical Price) − 1

For the following paragraph we reference this:

Nasdaq. (2024). Stock investing using momentum. Nasdaq.Com. https://www.nasdaq.com/articles/stock-investing-using-momentum

Our momentum calculation window ends one month before our price data does, which is a widely accepted standard in momentum strategies (Nasdaq, 2024). Over very short periods, stock prices tend to partially reverse. That is, a stock that rose sharply over the past month is slightly more likely to fall back the following month. Including that month would add noise that works against the longer-term trend we're trying to capture.

### Metric 3: Dividends paid

We find each dividend payment a stock makes during our sample period and divide it by the stock's price on the ex-dividend date to calculate what we call "dividend yield." Each of these yields is then weighted by how recent it is using a half life of 1 trading year. We then sum up these weighted yields over the sample period for each stock.

- Why we penalize dividends:
- Paying dividends is a signal for slower growth / maturity: as discussed in class, companies can optinally pay dividends. Companies that have investments / growth opportunities to seize (have better ways to use teh cash productively) reinvest into the business rather than paying dividends. A company paying dividends does so when it often has nothing better to do with the cash (than to appease investors). Thus, a company paying dividens is less likely to reinvest in the business for future growth, and is expected increase less in share price in the future, as they are not actively trying to grow the value of the company as much as other companies.

Why half life:
- because a company that paid dividends 3 years ago may not be doing so anymore today and should be penalized less than other companies that still do so today. If they were not trying to grow before but are trying to grow now, they should be penalized less (i.e. dividends from longer ago should be considered / weighted less, thus use half life).

However, this can't be used as a perfect indicator that a company is growing. For example, most tech firms aren't expected to pay out dividends, while most banks consistently pay out dividends. This is why we used it, but with a lower weight. 

### Metric 4: Sharpe


It's commonly used to measure risk-adjusted returns, and to see if a stock can outperform a risk free investment in government bonds. The momentum/return alone doesn't tell us how much risk the company took on to earn that return, which Sharpe tells us (if we took on a fair amount of risk to earn that return). We calculated it as:

> (Return - risk free rate) / standard deviation

For each stock, we take its daily returns over our sample and find the average return, and standard deviation, giving recent days more weight so the result using half life of 1 trading year, to reflect more how the stock behaves today. We calculate the same average return for a treasury bill ETF (CBIL.TO), which is our risk-free rate.

After calculating Sharpe, we only consider it if it's negative (if it's positive it gets set to 0). A negative Sharpe means the stock didn't even beat risk-free bonds, so these stocks are penalized in proportion to how far below 0 they are (so a stock near 0 Sharpe isn't penalized as hard as a really poor Sharpe stock). 

We don't reward stocks that pass, since a high Sharpe ratio can come from low volatility, not just high returns. This is already accounted for in our momentum and revenue metrics, so Sharpe is only used to filter our stocks whose returns weren't worht their risk in the first place. 

### Metric 5: Revenue CAGR

This measures how fast a company's yearly revenue (not price) grows on average each year. We use yfinance to pull financial statements from the past 3 years for this.

Calculation source:
Martin, V. (2025, February 11). Compound annual growth rate (CAGR) definition. Investing.Com. https://www.investing.com/academy/analysis/compound-annual-growth-rate-formula/

> CAGR = (latest yearly revenue / oldest yearly revenue) ^ (1 / number of years) - 1

Explanation source:
Brock, C. (2025, April 9). What is the compound annual growth rate? The Motley Fool. https://www.fool.com/terms/c/compound-annual-growth-rate/

We use CAGR instead of a simple average since average annual return tends to overstate actual growth and CAGR is a more realistic and conservative figure (Brock, 2025). This happens because each year's percentage change is measured from a different starting amount for that year. CAGR only uses the start and end values of revenue, giving a steady yearly growth rate.

We used this to measure revenue since that shows the growth of the fundamental business itself unrelated to investor/market sentiment or speculation. If revenue isn't growing, then there isn't much room for profit of the business to grow either, which influences stock price. Also, revenue is important for considering early stage companeis that may have growing revenues, but are still dealing with early-stage expenses that may subside in the future, leading to a high rapid growth stock.

### Metric 6: Revenue Growth STD

We measure how consistently a company's revenue grows year over year. While revenue CAGR measures how much it grew overall, it doesn't show the in-between risk or volatility in that growth. We calculate revenue growth std as follows:

We take the percentage change in revenue each year (i.e. the yearly growth rate), then take the standard deviation of those yearly growth rates. We consider 3 yearly growth rates, since requiring more data would leave too many stocks with missing values.

A low value means the company's revenue grows at a steady, predictable rate each year, which is more likely to continue. A high value means growth was more volatile, e.g. driven by a short burst of growth from one-off events that may not repeat in the future. It favours companies whose revenue is not just stable, but increasing year after year at a steady pace.

This metric gets a lower weight since it's based on only 3 yearly growth rates, which isn't a very large sample.

### Normalizing with z-score

Since a lot of the values we're getting are in different units and scales, the values from one metric can be completely different from the values of another metric. To fix this, we convert each metric into a z-score

Source: 
Nevil, S. (2025, March 11). What a Z-score tells us. Investopedia. https://www.investopedia.com/terms/z/zscore.asp

> z-score = (stock's metric value - avg. value across all stocks) / standard deviation of metric across all stocks

The z-score gives a standardized measurement of how far the metric for a stock is from the mean, in terms of standard deviations, putting every metric onto the same comparable scale centered at z = 0. 

If a stock is missing data or a metric, it gets the worst z-score for that metric out of all other stocks (we assume the worst case, and work with what best stocks we can find from the rest). The exception is drops; we just give a penalty of 1.5 points per drop for that stock.

Overall growth score:

Growth score = −1.5 × (number of drops) +
1 × (momentum z-score) +
− 0.375 × (dividend yield z-score) +
0.4 × (Sharpe check z-score) +
0.875 × (revenue CAGR z-score) +
− 0.25 × (revenue growth std z-score)

#### Stage 3 - Weighting

We rank every eligible stock from highest to lowest growth score, go down the list, and pick stocks in order. We take at most 3 from each sector to diversify against the business risk of any single industry: if one industry has bad news, at most 3 of our stocks are affected so overall there should still be some growth / positive returns. We keep picking until we have 11 stocks. We then take the growth scores of these 11 stocks and shift them down until the 11th (lowest-ranked) stock reaches a score of 0, then run our weighting algorithm.

It assigns each stock a weight in proportion to its new shifted score, so a stock that scored higher gets more weight (due to its higher long-term growth potential). We assign weights starting from the highest-scoring stocks first, making sure no stock gets more than 15% and no sector goes above 40%. If a stock would go over either limit, we cap it (at 15%, or at whatever weightage is left in its sector), and the leftover weight is shared among the remaining stocks in proportion to their scores. In the end, we drop the 11th stock since it ends up with a weight of 0%, leaving our final 10 stocks. 

We wanted to end up with 10 stocks since that is the minimum allowed, and our stocks are picked in order of growth score. Every additional stock would have a lower score and dilutes our client's expected return. It would be decrease risk, but also decrease returns, which we don't want.

**What evidence supports your strategy? Use statistics and the equal-weight comparison below to evaluate your approach, including any meaningful trade-offs.**

For the equal weight comparison, both portfolios were evaluated over October 2, 2023 to September 30, 2026, using daily closing prices converted to CAD and adjusted for splits / dividends. Both are treated as a buy on the start date, held until the end date (no rebalancing of weights after intial purchase).

Our weighting gained 4.2 percentage points of return more than equal weighting over three years. Our portfolio returned 149.7% over the three years. The tradeoff was that our daily standard deviation was higher (1.27% vs 1.12%). This makes sense since the weighted portfolio puts more money into faster growing but more volatile stocks, concentrated in fewer stocks too. Per unit of risk, equal weighting does better, but Client A wants the highest expeted return, not risk-adjusted return, so we accept this.

We also outperformed the S&P500 / market during this period, which had returns of 86.54%, but a much lower standard deviation of 0.9941%. It's the same tradeoff as above, more risk for higher returns.

Importantly, we did not optimize our weights or stock picks to maximize returns alone over this period. These were chosen using financial fundamentals and signals for growth. This reduces the risk that our results are overfitted to this period, and that the portfolio perform similarly in other periods. However, since our momentum and SHarpe are based on this period, it's not guaranteed that the same results would occur in the future (i.e. historical results don't guarantee future performance).

**What could go wrong? Identify at least one important weakness or situation in which the strategy may perform poorly.**

- Not really much diversification: Although our sector cap limits each sector to 3 stocks, stocks within the same sector tend to be closely correlated, since they're affected by the same industry news and regulation. A single sector can still make up to 40% of the portfolio, so a shock in a sector could reduce returns on much of the portfolio. On top of this, our weighting concentrates investment into the top stocks which further risks the strategy if any of those face poor performance or business problems in the future. Even worse would be a marketwide downturn, since we didn't focus on reducing market risk.

- Raw company size missing: One factor that our portfolio doesn't consider is the raw company size through market cap. This means that a penny stock with high momentum could be chosen, butthere was no differentiation between a company with magnitudes higher revenue. Thus, if the portfolio ends up carrying too many penny stocks / startups, these funds are a lot more risky, and are much more likely to fail in the future, which could ruin portfolio growth. But from our stock sampels and testing, it doesn't seem like a large problem.

- Momentum can reverse sharply: we weight momentum the highest, but if the market sentiment shifts on large momentum stocks, historical high return stocks can turn into declinign stocks in the future. Since Client A can't divest, they would be exposed to the full decline.

## Client B — Low Risk

Construct a portfolio intended to achieve the **lowest risk**, defined as the standard deviation of daily portfolio returns in CAD. Create `Portfolio_Final_B` using the same DataFrame column names, order, and index convention as for Client A.

Step 1.
- compute stats to be used for each stock

In [ ]:
# approach to calculate mean, std, and beta, which gives equal weight to all data
# it is used for the low-risk portfolio selection only so far

# Get rid of market from new returns dataframe
daily_returns = aligned_returns[eligible_tickers]

# calculate the mean daily returns of each stock
mean_returns = daily_returns.mean()  

# calculate the standard deviation of daily returns of stocks
daily_std = daily_returns.std(ddof=1)  

# calculate beta for each stock:
market_var = aligned_returns["Market"].var()

# Calc and store beta for each valid stock
betas = {}
# loop throguh each valid ticker
for tick in eligible_tickers:
    # temporarily store this stock's daily returns and the market's daily returns side by side
    temp = aligned_returns[[tick, "Market"]]
    # beta = covariance of stock with market / variance of market ([0, 1] picks the beta calculation cell from the output covariance table)
    betas[tick] = (temp.cov() / market_var).iat[0, 1] 

# turn dict into a series (column to be added)
betas = pd.Series(betas)

# start the stats table with data for eligible stocks only
stock_stats = stock_info.loc[eligible_tickers].copy()
stock_stats["Mean Return"] = mean_returns
stock_stats["Std"] = daily_std
stock_stats["Beta"] = betas

# These are the dataframes with the statistics we need
display(daily_returns.iloc[:5, :5])
display(stock_stats.head())


In [ ]:
period = 5
# count a decline when ending price is at most 80% of the starting price
threshold = 0.8

In [ ]:
# Note: this shouldn't be used as a raw value, as there may be periods where the price
# crashes are spread across two periods and thus not counted. Use a threshold lower than
# what you're truly looking for. 

plummets = count_price_plummets(period, threshold)["Occurrences"]
# store the crash count for use in risk score
stock_stats["Crashes"] = plummets

In [ ]:
# market-cap threshold of 20 billion CAD for the small-cap flag
min_market_cap = 20000000000

In [ ]:
# get purchase date exchange rate for market cap conversion
usd_to_cad_today = yf.download("CAD=X", start=buy_start, end=day_after, auto_adjust=True)["Close"]["CAD=X"].iloc[0]

# check if each stock is small cap if its market cap in CAD is under our threshold
for tick in stock_stats.index:

    # get the stock's market cap
    cap = stock_info.loc[tick, "Market Cap"]

    # convert USD market caps to CAD
    if stock_info.loc[tick, "Currency"] == "USD":
        cap = cap * usd_to_cad_today

    # Store whether small cap or large cap
    stock_stats.loc[tick, "Small Cap"] = cap < min_market_cap

display(stock_stats)

Step 2. Rank all stocks using their daily std and beta.


In [ ]:
# calculate std z score and beta z score and add to the table above

penalty = 10 # large number just to push the stocks down from getting used

std_weight = 0.3
beta_weight = 0.7

std_z = (stock_stats["Std"] - stock_stats["Std"].mean()) / stock_stats["Std"].std()
beta_z = (stock_stats["Beta"] - stock_stats["Beta"].mean()) / stock_stats["Beta"].std()

# Combine z scores into a risk score in the dataframe
stock_stats["Risk Score"] = (std_weight * std_z) + (beta_weight * beta_z) + (penalty * stock_stats["Crashes"]) + (penalty * stock_stats["Small Cap"])

# Sort the dataframe, lower risk score is safer
stock_stats = stock_stats.sort_values("Risk Score")

# convert risk score sorted stocks into a list
ranked = stock_stats.index.tolist()

display(stock_stats)
display(ranked)

Step 3. Picking stocks

In [ ]:
# combined_std is a function that takes in a list of tickers
# it returns the std of the daily returns of an equal weighted portfolio comprised of thoe given stocks
def combined_std(tickers):

    # every stock gets equal weight
    weights = 1 / len(tickers)

    returns = 0
    
    for tick in tickers:
        returns = returns + weights * daily_returns[tick]
        
    # return standard deviation of daily portfolio returns
    return returns.std(ddof=1)

In [ ]:
# dict to store running tally of number of stocks from each sector
sector_tally = {}

# max number of stocks per sector: for diversification
max_per_sector = int(sector_limit * min_stocks)  

# list to store selected stocks for portfolio
selected = []

# initialize tally for each sector so far
for tick in ranked:
    sector_tally[stock_stats.loc[tick, "Sector"]] = 0

# start with the safest stock in selected
selected.append(ranked[0])

# add 1 to its sector tally
sector_tally[stock_stats.loc[ranked[0], "Sector"]] = 1

# total number of stocks in portfolio
total_count = 1

# std of current portfolio to compare to
current_std = combined_std(selected)

# start by only allowing stocks with risk scores below -0.5, then if not enough, we'll increase threshold in increments and continue picking
score_limit = -0.5
score_step = 0.1

while total_count < max_stocks:
    # starting std of portfolio in current iteration (whatever stocks are in selected + the best stock found so far down the ranked list)
    best_tick = None
    best_std = None

    # loop through safe ranked stocks top down
    for tick in ranked:
        # skip any stocks already chosen for portfolio
        if tick in selected:
            continue

        # find the current stock's sector
        sector = stock_stats.loc[tick, "Sector"]
        # temporarily add 1 to that sector's tally for a check
        temp_tally = sector_tally[sector] + 1

        # if adding another stock from that sector goes over the limit, skip that stock
        if temp_tally > max_per_sector:
            continue

        # skip stocks where risk score is above the current limit
        if stock_stats.loc[tick, "Risk Score"] > score_limit:
            continue
            
        # calculate std if this stock were to be added to portfolio
        new_std = combined_std(selected + [tick])

        # skip the stock if the new std is worse
        if best_std is not None and new_std >= best_std:
            continue

        # if the stock passed all the tests
        best_tick = tick
        best_std = new_std

    # no stock fits requirements to be added to portfolio
    if best_tick is None:

        # if we're still under minimum num of stocks, raise the risk score by a little and continue picking stocks
        # check if we need more stocks, and that we haven't already maxed out our score limit (theoretically won't matter)
        if total_count < min_stocks and score_limit <= stock_stats["Risk Score"].max():
            score_limit += score_step
            continue
        break

    # otherwise: adding the stock lowers std, or we still need more stocks to reach the minimum
    selected.append(best_tick)
    sector_tally[stock_stats.loc[best_tick, "Sector"]] += 1
    total_count += 1
    current_std = best_std

print("Selected:", selected)
print("Total stocks:", total_count)
print("Stocks per sector:", sector_tally)
print("Equal-weight portfolio daily std:", current_std)

Step 4. Weight Optimization:
- using iterative approach


In [ ]:
# Weight Optimization

# convert selected stock list into a series, alongside equal weight to each stock
initial_weights = pd.Series(1 / len(selected), index = selected)

# create a series, sectors, to store the stock name alongside its sector
sectors = {}
for tick in selected:
    sectors[tick] = stock_stats.loc[tick, "Sector"]

sectors = pd.Series(sectors)

# create a minimum weight each stock must conserve (otherwise we might get stocks with 0% after giving away 1%, or just neglibible stocks that don't do anything)

min_weight = 0.02

# from canada CPI, latest inflation rate
inflation = 0.031 

# calculating each stock's average year over year return, into a Series, for inflation comparison
yearly_returns = stock_stats.loc[selected, "Mean Return"] * 252

# will loop through the weight optimizer on 1 year periods, which occur every month
hold_days = 252 # roughly 1 year
start_every = 21 # roughly 1 trading month

# optimize_weights is a function that takes in a dataframe of stock closing prices for a certain trading period
# it returns the weights for each stock in the portfolio, that achieve the lowest std found by shifting 1% between stocks within the limits
def optimize_weights(period_prices):

    # For each stock try transferring 1% point to every other stock and keep the lowest std
    
    # start from equal weights
    target_weights = initial_weights.copy() 
    # step is the amount of weight we want to shift between stocks
    step = 0.01  
    # std of the starting equal weight portfolio
    current_std = weighted_std(target_weights, period_prices)  
    # history of each move made (donor, receiver, new std of that portfolio)
    move_history = []
    
    improved = True  
    while improved: 
        improved = False  # Assume no improvement at the beginning of each iteration
    
        for a in selected:  # Try each selected stock as the donor
            # skip the stock if donating x% puts it under the min weight requirement
            if target_weights[a] - step < min_weight:
                continue

            # track the best recipient for the current donor stock
            best_b = None  
            lowest_std = current_std  
    
            for b in selected:  
                if a == b:  
                    continue

                # test a transfer without changing the current portfolio
                trial = target_weights.copy()  
                trial[a] -= step  
                trial[b] += step 

                # reject transfers that exceed the weight cap
                if trial.max() > stock_limit: 
                    continue
                # reject transfers that exceed the sector weight cap
                if trial.groupby(sectors).sum().max() > sector_limit:  
                    continue
                   

                # multiply stock weights by their yearly returns to get stock contribution to portfolio, add it all up to get portfolio return
                if (trial * yearly_returns).sum() < inflation:
                    continue

                # calculate portfolio std after the weight transfer
                trial_std = weighted_std(trial, period_prices)  

                # keep the lowest volatility recipient so far
                if trial_std < lowest_std:
                    best_b = b
                    lowest_std = trial_std

            # apply the best improving transfer for the donor
            if best_b is not None:  
                target_weights[a] -= step
                target_weights[best_b] += step
                current_std = lowest_std
                move_history.append([a, best_b, current_std])  
                improved = True

    # return the final weights and volatility
    return target_weights, current_std

# a list that stores the optimzied weights for each 1 year period tested
all_timelines = []
stds = []

# start a new 1 year window every month, stopping 1 year before the end of our data so every window being tested has a full year of prices
for i in range(0, len(aligned_prices) - hold_days, start_every):
    selected_stocks_prices = aligned_prices[selected] # toss away price data for stocks we didn't select
    start_row = i # 0th index row
    end_row = i + hold_days # last row we want in the timeframe when running , 

    # take rows in the current trading window being considered
    #note that iloc only includes (end_row - start_row number of rows) so we add 1
    trading_window = selected_stocks_prices.iloc[start_row:(end_row + 1)] 
    
    # optimize weights for this the 1 year trading window
    weights, std = optimize_weights(trading_window)
    # save this window's optimized weights and std to the list, all_timelines
    all_timelines.append(weights)
    # add the windows' optimized std, to the list stds, to avg across all windows later for reporting
    stds.append(std)
    
# as of now, all_weights is a list containing Pandas series for each set of optimized weights from each period tested, attributed to each stock by name

# average the optimized weights found across all periods tested

print(f"Periods tested: {len(all_timelines)}")

# add up all the weights from every single series across all series, then divide each one by the number of series in total 
# i.e. avg the weights for each stock, to find ideal weighting
weights_B = sum(all_timelines) / len(all_timelines)

print(f"Average Optimzied Std: {np.mean(stds)}")  # average standard deviation after weight optimization

display(weights_B)

## Outputting Final Portfolio

In [ ]:
# send all data to function to create final dataframe
Portfolio_Final_B, fees_B, total_cost_B = final_table(weights_B.to_dict(), budget_B)

# show the final Client B holdings and weights
display(Portfolio_Final_B)

# Run the portfolio checks using Client B's actual costs and budget
validate_portfolio(Portfolio_Final_B, fees_B, total_cost_B, budget_B)

### Client B — Financial justification
The code right below is for generating the equal weight portfolio for the same stocks chosen and extra stats to compare

In [ ]:
# compare the low volatility weights against equal weights for the same tickers
compare_equal_weight(weights_B)

In [ ]:
# create a correlation matrix of all the selected stocks
corr_mat = pd.DataFrame()

# go through every combination of stocks
for t1 in selected:
    for t2 in selected:
        # calcualte the correlation between daily returns of both stocks
        corr_mat.loc[t1, t2] = daily_returns[t1].corr(daily_returns[t2])

display(corr_mat.round(3))

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.


Our stock picking strategy follows these stages:

#### Stage 1 - Market Stats

We download data of the S&P 500 over the same period as our stock data. The closing prices of index represent the market and are used to calculate beta

The period chosen for our historical sampling/data was: 2023-10-02 to 2026-09-30
We chose this because the client wants a long-term, low-risk portfolio. This spans 3 years, and so gives a longer-term picture for statistics we will be using. Also, going further back, we noticed that some stocks were missing data we needed, some may not exist, and their businesses may not be similar to what they are today. 3 years is a good middle ground for how much data we can consider, and it's the same justification for client A's portfolio.

We chose the S&P500 since it is a very broad/diversified, market-cap-weighted index. It also represents the Canada and US markets overall pretty well (where all the stocks are coming from): while all S&P500 stocks are US-based, the US market makes up the majority of the equity value in these two countries, so it is a reasonable choice, the same justification as for Client A's portfolio.

#### Stage 2 - Statistics for every stock

For every eligible stock we calculate 4 statistics:
- Standard deviation of daily returns
- Beta of the stock's daily returns vs. market returns
- Crash count: how many times the stock fell by 20% in price within 5 day trading periods, over our data period
- Market cap: measured in CAD on the day trading starts

### Metric 1: Standard deviation of daily returns

This is what the client uses to define risk, so we start with this calculation. It is by definition a measure of the stock's volatility in its returns, where we used daily returns to increase the number of data points we have over a smaller historical period (since going too far back means businesses may not be similar to what they are today, making statistics misleading).

### Metric 2: Beta

Beta is calculated just like in class, as the covariance of the stock's returns with the market divided by the market's variance. Instead we are using daily returns and with the S&P 500 converted to CAD so it matches our stock prices.

While we can diversify our stocks to remove industry or business risk, this still would not remove market risk since all the stocks chosen are subject to shocks/volatility of the underlying market. So, to minimize the impacts of market risk, we focus on picking stocks with low beta (i.e. when the market moves in a direction, these stocks move less in that direction, compared to others). This is important for our client since they are buying and holding, so in a market downturn, they cannot exit their position, so we need our stock to minimize losses to maintain stability in this potential situation.

### Metric 3: Number of drops

This is a future-predictive statistic we created. While std already looks at volatility, we specifically want to capture company risks that decrease value quickly. If a stock loses 20% within a week due to company-specific news or shocks, such as earnings reports, similar events could happen again in the future. This accounts for non-market / business risk.

We split our sample into 5 trading day periods (about 1 trading week) and count how many times the stock fell by 20% or more within one of those periods. Stocks that dropped suddenly more often in the past have faced more company-specific news or shocks, such as earnings reports or regulation.

If the market as a whole declines sharply within 1 week in our sample, many stocks are penalized at once, so it doesn't change the rankings much. However, if a specific stock or industry declines sharply while others don't, our penalty helps differentiate which stocks to pick. This directly captures non-market risks like poor management decisions or shocks from news, which can characterize certain stocks and industries since they occur more frequently for them than for others.

### Metric 4: Market Cap

Large and mature companies often have steady revenue streams, earnings, and are usually stabler than penny stocks. Specifically, we don't want to invest in a penny stock, startup, or similar company, that is simply stagnant or has low volume. On paper, these stocks may have low volatility, but their price could change dramatically in the future, often because these firms are more likely to go bankrupt / fail as businesses. So, instead we focus on established businesses that are less likely to face such events.

#### Stage 3 - Risk score ranking

Each stock is given a risk score, lower is safer.

> Risk score = 0.3 * (std z score) + 0.7 * (beta z-score) + 10 × (number of crashes) + 10 × (1 if market cap < $20B CAD)

Stocks are sorted from safest to riskiest.

#### Normalizing with z-score

Since a lot of the values we're getting are in different units and scales, the values from one metric can be completely different from the values of another metric. To fix this, we convert each metric into a z-score

Source: 
Nevil, S. (2025, March 11). What a Z-score tells us. Investopedia. https://www.investopedia.com/terms/z/zscore.asp

> z-score = (stock's metric value - avg. value across all stocks) / standard deviation of metric across all stocks

The z-score gives a standardized measurement of how far the metric for a stock is from the mean, in terms of standard deviations, putting every metric onto the same comparable scale centered at z = 0. 

#### Beta weighted more than standard deviation

Beta gets the higher weight (0.7 vs 0.3) because it targets market risk, which diversification can't remove itself. Standard deviation is already reduced later on by our picker and weight optimizer.

#### Dispraportionate penalties on stocks for having sudden drops or for being a small cap stock

The penalties are just arbitrarily large numbers to guarantee (most of the time) that a crashed or small-cap stock ranks way below the rest of the safer stocks, and will most likely be discluded from our ultimate portfolio. We only penalize the stocks, rather than dropping them from our list, in case future stages still need more stocks to meet minimum portfolio requirements (eg. 10 stock minimum), but these stay on only as a last resort choice.

#### Stage 4 - Stock selection algorithm

**What it does:** We start with the stock that has the lowest risk score. Then, one stock at a time, we add whichever remaining stock gives the lowest standard deviation for the new combined portfolio. Only stocks with a risk score below −0.5 (well below average risk) are considered at first, and at most 4 stocks can be coming from from any one sector. The picker keeps adding stocks until no more stocks pass these limits. If that results in less than 10 stocks, the risk score cutoff is loosened in steps of 0.1, so riskier stocks can be considered slowly, and the reiskiest stocks are only considered as a last resort to meet portfolio requirements.

**Why these choices:**
- Choosing by portfolio std: std of a portfolio is not equal to the average of the stocks individual std. It is usually less than this, which is the benefit of diversification, and this idea leads the whole picking process. How much lower depends on how the stocks move together (correlation): if one stock tends to rise when another falls, for instance if their industries react differently same news, their movements partly cancel out. By measuring the standard deviation of the combined portfolio at each step, the picker captures the benefits of these offsetting relationships, which cannot be seen just by looking at individual statistics of each stock alone.
- Equal weighting combined portfolios: At each step, we compute the std of an equal-weighted portfolio of the stocks chosen so far, plus the candidate. Early on, each new stock is a large share of this small portfolio, so the picker favours stocks that are stable on their own, setting up a core of stable stocks as a baseline. As more stocks are added, each new stock's share shrinks, so the picker focuses more on how the candidate correlates with the ultra safe stocks already chosen, to further reduce standard deviation.
- Starting from lowest risk scored stock: as seen in class, adding a riskier stock to a stabler stock in a portfolio can raise the std of the overall portfolio, from the std of the stabler stock alone. To account for this, we start off with the lowest risk stock and add stocks one at a time, choosing the stock which gives us the lowest combined portfolio std at each round. Sometimes this increases or decreases the combined portfolio's std, but it is always the best option available at that step.
- Score cutoff of -0.5: Testing with the picker, we found that without a cutoff, it picks the riskiest stocks because they often offset the calmer ones. We want to avoid choosing these risky stocks too early, so we restrict the picker to safer ones first, with safer ones still possibly offsetting each other. Gradually loosening it ensures at least 10 stocks are chosen.
- 4 stocks per sector: In the most restrictive case, we want at most 40% of 10 stocks in any sector. So this cap helps meet portfolio requirements (40% of 10 = 4). It also forces the portfolio to hold across industries and protect against sector-specific events/shocks (eg., regulation changes, outlook of an industry, etc.). This is added diversification and mitigation of business / non-systemic risk. 

#### Stage 5 - Weight optimization

**What it does:** Starting from equal weights, the optimizer tries moving 1% weight from one stock to another. It makes the shift if it lowers overall portfolio volatility, theoretically if it was held over a 1 year period in the past. It keeps on doing this until the weights minimize volatility, computing the optimal weights for that 1 year hold. Then, it shifts by 1 month, to consider a new 1 year period, and does the same thing. Throughout the process, each stock keeps at least 2% weight, no stock can exceed 15%, and no sector can exceed 40%. If the weight change makes the portfolio's annual return (average over our sample) less than inflation rate, it rejects that change. In the end, the average weights for each stock, as optimzied for every 1 year period, is computed, giving final weights.

**Why these choices:**
- Trying many 1-year periods: if we optimized for a single 1 year period, we would overfit to those events, and it would not reflect how the portfolio performs in the future, which has different events. Our strategy optimizes weights to work well across many different long-term periods.
- 2% minimum weight: Prevents weights that completely eradicate stocks, by giving meaningless weights near 0%, and overly relying on a few core stocks. Keeping some weighting in these stocks, rather than 0%, also helps maintain some diversification.
- Comparison to inflation rate: Client B wants to conserve as much money as possible, over a long period of time. If they just held completely unvolatile low growth stocks, which we try to achieve, they will still have a negative real rate of return, i.e. they lose money due to inflation over time. So if theoretically a portfolio with 0 volatility could be obtained, it still wouldn't help conserve the value of Client B's money, being as useless as cash at conserving it. So, we ensure that our weighted portfolio can beat inflation rate in our historical sample to increase the chance that it can do this in the future. In our tests we found that our portfolio anyways had really high nominal returns due to markets / large cap stocks performing well, so this may not make much of an impact. However, if a trul low volatility portfolio is achieved with new tickers, real returns should be kept positive.


**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

For the equal weight comparison, both portfolios were evaluated over October 2, 2023 to September 30, 2026, using daily closing prices converted to CAD and adjusted for splits / dividends. Both are treated as a buy on the start date, held until the end date (no rebalancing of weights after intial purchase).

Our weighting lowered the daily standard deviation from 0.74% to 0.65% compared to equal weighting, which is about 11% less volatile. Since Client B defines risk as the standard deviation of daily returns, this shows our weighting helps achieve the client's goal. There was no trade-off in return either which was surprising to us. Our portfolio returned 72.8% over the three years vs 59.6% for equal weighting, which is 13.1 percentage points more. This makes sense since our optimizer moved weight away from the more volatile stocks, and in this period those weren't the best performers either. But we wouldn't expect this in every period or in the future. Normally we'd expect a trade-off, especially with a ticker list that includes more volatile, higher-growth stocks.

Looking at the correlation matrix of our selected stocks, none of the pairs had negative correlations. However, the portfolio includes several pairs with near-zero correlation, such as T.TO and LMT (0.012), RY.TO and MO (0.013), and TD.TO and MO (0.017). Since these stocks barely move together, their price swings partly cancel out, which lowers the portfolio's standard deviation further, showing that our picker captures this effect as intended.

Importantly, we did not optimize our weights to target one period. Instead we optimized them separately over 24 different overlapping one-year windows and averaged the results, so the weights aren't tuned to one specific period. This reduces the risk that our results are overfitted to one singular period, which makes it more likely the portfolio has low voaltility in other periods. We also chose our stocks using beta, past crashes and market cap, not standard deviation alone. However, since our statistics and weights all come from this same historical data, historical results don't guarantee future performance.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.

- The past may not represent the future: all of our stats come from 1 period of past data, for simplification. For instance, the data does not include a full market crisis like 2008, which could be good for informing how this portfolio would perform in such a sever downturn, as our client wants a long-term portfolio and strategy. Our beta-based ranking can reduce this risk but cannot completely remove market risk.

- Concentration in sectors: Our portfolio in testing ended up with many stocks in consumer staples, healthcare, large banks, and telecom. For example, we hold three Canadian stocks, T.TO, RY.TO, TD.TO, which make up about 43% of the portfolio and all depend on the Canadian economy and interest rates, so a Canadian market downturn would cause declines in all of them together. Similarly, our healthcare stocks (ABT, PFE, ABBV, MRK) could all be hurt by the same event.

- Many of our holdings are US stocks, so their CAD value depends on the USD/CAD exchange rate. An increase in the Canadian dollar would lower the CAD value of these positions even if their US prices didn't change.

- Crash potential: a stock that has never plummeted quickly in the past still has the potential to do so in the future, and any of the stocks chosen could plummet in the future unexpectedly.

- Equal weight selection: Our picker algorithm uses equal weights to decide stocks to combine into portfolio next. This may overlook stocks that can outperform others at a non-equal weight, or can deliver a better opposite directional benefit when weighted lower amount. But realistically, it would be really hard to implement dynamic weight considerations here, given portfolio restrctions and an unkown number of stocks we will have in the final portfolio.

The following team members made a meaningful contribution to this assignment:

- Brian Zhang
- Vansh Chauhan
- Jenny Liu

To the best of our knowledge, no generative AI was used in preparing this submission.